In [1]:
pip install pinecone

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.6 MB 6.9 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.6 MB 4.6 MB/s eta 0:00:01
   ------------------------------------ --- 2.4/2.6 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 3.7 MB/s  0:00:00

   -------------------------- ------------- 2/3 [pinecone]
   -------------------------- ------------- 2/3 [pinecone]
   -------------------------- ------------- 2/3 [pinecone]
   -------------------------- ------------- 2/3 [pinecone]
   -------------------------- ------------- 2/3 [pinecone]
   -------------------------- ------------- 2/3 [pinecone]
   -------------------------- ------------- 2/3 [pinecone]
   -------------------------- ------------- 2/3 [pinecone]
   -------------------------- ------------- 2/3 [pinecone]
   ---------------------------------------- 3/3 [pinecone]

Note: you may need to restart the kern


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pinecone import Pinecone, ServerlessSpec

In [3]:
documents = [
    {
        "id": "doc-001",
        "text": "Pinecone is a fully managed vector database for search and recommendation.",
        "category": "documentation",
        "tag": "pinecone",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-intro"
    },
    {
        "id": "doc-002",
        "text": "To use Pinecone with Python, you create an index and upsert vectors with metadata.",
        "category": "documentation",
        "tag": "python",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-python"
    },
    {
        "id": "doc-003",
        "text": "Vector databases store embeddings that capture semantic meaning for semantic search.",
        "category": "blog",
        "tag": "vector-db",
        "difficulty": "intermediate",
        "url": "https://example.com/vector-db-concepts"
    },
    {
        "id": "doc-004",
        "text": "You can filter Pinecone search results using metadata such as category or difficulty.",
        "category": "faq",
        "tag": "metadata",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-metadata"
    },
    {
        "id": "doc-005",
        "text": "In Retrieval-Augmented Generation, a vector database like Pinecone stores document chunks.",
        "category": "blog",
        "tag": "rag",
        "difficulty": "intermediate",
        "url": "https://example.com/rag-pinecone"
    }
]

In [4]:
pc = Pinecone(api_key="pcsk_4PK2AD_EYTK6Fz6MCFvhRaHupR2AurRzjHzMBeAsutR4mpgSnvpcgjQw67Lo2MNkgapgbr")

In [5]:
pc

Pinecone(api_key='...pgbr', host='https://api.pinecone.io')

In [6]:
import requests
import numpy as np
from typing import List, Union
import os

EURON_API_KEY = "euri-87b18ab55074ec551bf13230f03ebd05d4cea02bd2dc9d959edf9117744f8461"

def generate_embeddings(texts: Union[str, List[str]]):
    if isinstance(texts, str):
        texts = [texts]

    url = "https://api.euron.one/api/v1/euri/embeddings"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {EURON_API_KEY}"
    }

    payload = {
        "input": texts,
        "model": "text-embedding-3-small"
    }
    response = requests.post(url,headers=headers,json=payload,timeout=30)
    data = response.json()

    embeddings = [np.array(item["embedding"], dtype=np.float32) for item in data["data"]]

    return embeddings[0] if len(embeddings) == 1 else np.stack(embeddings)

In [ ]:
index_name = "developer-quickstart-py"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"text-embedding-3-small",
            "field_map":{"text": "chunk_text"}
        }
    )

In [7]:
INDEX_NAME = "euron-pinecone-euri-demo"

In [8]:
pc.list_indexes()

IndexList([])

In [10]:
pc.create_index(
    name=INDEX_NAME,
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(
      cloud="aws",
      region="us-east-1"
    ),
)

IndexModel(name='euron-pinecone-euri-demo', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://euron-pinecone-euri-demo-oe6lqtb.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=1536, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [11]:
documents

[{'id': 'doc-001',
  'text': 'Pinecone is a fully managed vector database for search and recommendation.',
  'category': 'documentation',
  'tag': 'pinecone',
  'difficulty': 'beginner',
  'url': 'https://example.com/pinecone-intro'},
 {'id': 'doc-002',
  'text': 'To use Pinecone with Python, you create an index and upsert vectors with metadata.',
  'category': 'documentation',
  'tag': 'python',
  'difficulty': 'beginner',
  'url': 'https://example.com/pinecone-python'},
 {'id': 'doc-003',
  'text': 'Vector databases store embeddings that capture semantic meaning for semantic search.',
  'category': 'blog',
  'tag': 'vector-db',
  'difficulty': 'intermediate',
  'url': 'https://example.com/vector-db-concepts'},
 {'id': 'doc-004',
  'text': 'You can filter Pinecone search results using metadata such as category or difficulty.',
  'category': 'faq',
  'tag': 'metadata',
  'difficulty': 'beginner',
  'url': 'https://example.com/pinecone-metadata'},
 {'id': 'doc-005',
  'text': 'In Retrie

In [12]:
texts = [doc['text'] for doc in documents]

In [13]:
texts

['Pinecone is a fully managed vector database for search and recommendation.',
 'To use Pinecone with Python, you create an index and upsert vectors with metadata.',
 'Vector databases store embeddings that capture semantic meaning for semantic search.',
 'You can filter Pinecone search results using metadata such as category or difficulty.',
 'In Retrieval-Augmented Generation, a vector database like Pinecone stores document chunks.']

In [14]:
doc_embeddings = generate_embeddings(texts)

In [15]:
doc_embeddings

array([[-0.00132942,  0.00098705,  0.02589417, ...,  0.0103302 ,
        -0.00441742, -0.00444794],
       [ 0.02902222,  0.02787781,  0.04516602, ..., -0.01396942,
         0.004776  ,  0.01304626],
       [-0.02900696,  0.02354431,  0.01060486, ...,  0.0275116 ,
         0.00490189,  0.00447464],
       [ 0.02288818, -0.01055908,  0.05374146, ..., -0.00687027,
         0.02050781, -0.01083374],
       [ 0.01342773,  0.02363586,  0.0447998 , ..., -0.00083542,
         0.04611206,  0.01820374]], shape=(5, 1536), dtype=float32)

In [16]:
documents, doc_embeddings

([{'id': 'doc-001',
   'text': 'Pinecone is a fully managed vector database for search and recommendation.',
   'category': 'documentation',
   'tag': 'pinecone',
   'difficulty': 'beginner',
   'url': 'https://example.com/pinecone-intro'},
  {'id': 'doc-002',
   'text': 'To use Pinecone with Python, you create an index and upsert vectors with metadata.',
   'category': 'documentation',
   'tag': 'python',
   'difficulty': 'beginner',
   'url': 'https://example.com/pinecone-python'},
  {'id': 'doc-003',
   'text': 'Vector databases store embeddings that capture semantic meaning for semantic search.',
   'category': 'blog',
   'tag': 'vector-db',
   'difficulty': 'intermediate',
   'url': 'https://example.com/vector-db-concepts'},
  {'id': 'doc-004',
   'text': 'You can filter Pinecone search results using metadata such as category or difficulty.',
   'category': 'faq',
   'tag': 'metadata',
   'difficulty': 'beginner',
   'url': 'https://example.com/pinecone-metadata'},
  {'id': 'doc-0

In [21]:
vector_to_upsert = []
for doc, emb in zip(documents, doc_embeddings):
  print(doc, emb)
  metadata = {
    "category": doc["category"],
    "tag": doc["tag"],
    "difficulty": doc["difficulty"],
    "url": doc["url"],
    "text": doc["text"]
  }
  vector_item = {
    "id": doc["id"],
    "values": emb.tolist(),
    "metadata": metadata
  }
  vector_to_upsert.append(vector_item)

{'id': 'doc-001', 'text': 'Pinecone is a fully managed vector database for search and recommendation.', 'category': 'documentation', 'tag': 'pinecone', 'difficulty': 'beginner', 'url': 'https://example.com/pinecone-intro'} [-0.00132942  0.00098705  0.02589417 ...  0.0103302  -0.00441742
 -0.00444794]
{'id': 'doc-002', 'text': 'To use Pinecone with Python, you create an index and upsert vectors with metadata.', 'category': 'documentation', 'tag': 'python', 'difficulty': 'beginner', 'url': 'https://example.com/pinecone-python'} [ 0.02902222  0.02787781  0.04516602 ... -0.01396942  0.004776
  0.01304626]
{'id': 'doc-003', 'text': 'Vector databases store embeddings that capture semantic meaning for semantic search.', 'category': 'blog', 'tag': 'vector-db', 'difficulty': 'intermediate', 'url': 'https://example.com/vector-db-concepts'} [-0.02900696  0.02354431  0.01060486 ...  0.0275116   0.00490189
  0.00447464]
{'id': 'doc-004', 'text': 'You can filter Pinecone search results using metadat

In [22]:
vector_to_upsert

[{'id': 'doc-001',
  'values': [-0.0013294219970703125,
   0.0009870529174804688,
   0.0258941650390625,
   0.00646209716796875,
   0.0272979736328125,
   4.273653030395508e-05,
   -0.0019664764404296875,
   0.05804443359375,
   0.008941650390625,
   -0.00664520263671875,
   0.027557373046875,
   -0.00858306884765625,
   0.0205230712890625,
   -0.07861328125,
   0.0141448974609375,
   -0.0181427001953125,
   -0.028839111328125,
   0.0074005126953125,
   0.033294677734375,
   0.0390625,
   0.015777587890625,
   0.060516357421875,
   0.0291290283203125,
   -0.0181732177734375,
   0.0179290771484375,
   0.05926513671875,
   -0.033599853515625,
   0.04156494140625,
   0.08160400390625,
   -0.057525634765625,
   0.0217132568359375,
   -0.0230712890625,
   -0.0245208740234375,
   -0.00848388671875,
   0.0275115966796875,
   -3.0100345611572266e-05,
   -0.003955841064453125,
   0.016143798828125,
   -0.00727081298828125,
   0.023895263671875,
   -0.01788330078125,
   0.044189453125,
   0.0046

In [24]:
index = pc.Index(INDEX_NAME)

In [25]:
index

Index(host='https://euron-pinecone-euri-demo-oe6lqtb.svc.aped-4627-b74a.pinecone.io')

In [26]:
index.upsert(vectors=vector_to_upsert)

UpsertResponse(upserted_count=5)

In [ ]:
query_text = "How do I use Pinecone with Python?"
query_embedding = generate_embeddings(query_text)
pi_response = index.query(
  vector=query_embedding.tolist(),
  top_k=3,
  include_metadata=True,
  filter={ #optional filter to narrow down results based on metadata
    "difficulty": {"$eq": "beginner"}
    }
)

In [30]:
pi_response

QueryResponse(matches=[ScoredVector(id='doc-002', score=0.695411086, values=[], metadata={'category': 'documentation', 'difficulty': 'beginner', 'tag': 'python', 'text': 'To use Pinecone with Python, you create an index and upsert vectors with metadata.', 'url': 'https://example.com/pinecone-python'}), ScoredVector(id='doc-001', score=0.560460269, values=[], metadata={'category': 'documentation', 'difficulty': 'beginner', 'tag': 'pinecone', 'text': 'Pinecone is a fully managed vector database for search and recommendation.', 'url': 'https://example.com/pinecone-intro'}), ScoredVector(id='doc-004', score=0.486193597, values=[], metadata={'category': 'faq', 'difficulty': 'beginner', 'tag': 'metadata', 'text': 'You can filter Pinecone search results using metadata such as category or difficulty.', 'url': 'https://example.com/pinecone-metadata'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Mon, 10 Aug 2026 18:27:01 GMT', 'co